In [2]:
import BioSimSpace as bss
from pathlib import Path
import glob
import os
import pandas as pd

INFO:rdkit:Enabling RDKit 2025.03.4 jupyter extensions
INFO:numexpr.utils:NumExpr defaulting to 12 threads.


# Set repo path

This is only relevant for this example, in your case you will setup the path to your inputs. 

In [3]:
REPO_ROOT = Path().resolve()
DATA_DIR = REPO_ROOT / "data"

In [10]:
protein_inputs = os.path.join(
    DATA_DIR, "inputs", "protein"
)

ligand_inputs = os.path.join(
    DATA_DIR, "inputs", "ligands"
)

# 1. Load protein into BioSimSpace

In [7]:
protein_file = os.path.join(protein_inputs, "kpc2.prepared.pdb")

protein_molecule = bss.IO.readPDB(id=protein_file, pdb4amber=False, work_dir=protein_inputs)[0]


# 2. Parameterise protein

In more complicated cases it may be better to carry out the parameterisation in Amber/GROMACS and then load in the topology and coordinate files. 

In [9]:
parameterised_protein = bss.Parameters.parameterise(
    molecule=protein_molecule, forcefield="ff14SB", work_dir=protein_inputs
).getMolecule()

parameterised_protein_file = os.path.join(protein_inputs, "protein")
bss.IO.saveMolecules(filebase=parameterised_protein_file, system=parameterised_protein, fileformat=["prm7", "rst7"])

['/Users/af25016/projects/ligand_rbfe/data/inputs/protein/protein.prm7',
 '/Users/af25016/projects/ligand_rbfe/data/inputs/protein/protein.rst7']

# 3. Combine protein and ligand

First load in the parameterised ligand files:

In [12]:
ligand_1_directory = os.path.join(ligand_inputs, "ligand_1")
ligand_2_directory = os.path.join(ligand_inputs, "ligand_2")

parameterised_ligand_1 = bss.IO.readMolecules(
    [f"{ligand_1_directory}/ligand_1_gaff2.prm7",
     f"{ligand_1_directory}/ligand_1_gaff2.rst7"]
)

parameterised_ligand_2 = bss.IO.readMolecules(
    [f"{ligand_2_directory}/ligand_2_gaff2.prm7",
     f"{ligand_2_directory}/ligand_2_gaff2.rst7"]
)


Combine with the protein:

In [13]:
system_1 = parameterised_protein + parameterised_ligand_1
system_2 = parameterised_protein + parameterised_ligand_2


# 4. Create box

In [14]:
box_axis_length = 12.0 #nanometers

box_1_min, box_1_max = system_1.getAxisAlignedBoundingBox()
box_2_min, box_2_max = system_2.getAxisAlignedBoundingBox()


In [15]:
box_1_size = [y - x for x, y in zip(box_1_min, box_1_max)]
box_2_size = [y - x for x, y in zip(box_2_min, box_2_max)]
box_1_sizes = [x + int(box_axis_length) * bss.Units.Length.nanometer for x in box_1_size]
box_2_sizes = [x + int(box_axis_length) * bss.Units.Length.nanometer for x in box_2_size]


In [16]:
box_1, angles_1 = bss.Box.cubic(max(box_1_sizes))
box_2, angles_2 = bss.Box.cubic(max(box_2_sizes))

# 5. Solvate

In [17]:
system_1_directory = os.path.join(protein_inputs, "bound_ligand_1")
system_2_directory = os.path.join(protein_inputs, "bound_ligand_2")

# This will raise an error if the directory already exists, to avoid overwriting files. You can set exist_ok=True:
os.makedirs(system_1_directory, exist_ok=False)
os.makedirs(system_2_directory, exist_ok=False)



In [18]:
system_1_solvated = bss.Solvent.solvate(
    model="tip3p", 
    molecule=system_1, 
    box=box_1, 
    angles=angles_1, 
    ion_conc=0.15,
    work_dir=system_1_directory
)

In [21]:
system_2_solvated = bss.Solvent.solvate(
    model="tip3p", 
    molecule=system_2, 
    box=box_2, 
    angles=angles_2, 
    ion_conc=0.15,
    work_dir=system_2_directory
)

# 5. Save solvated systems

In [23]:
system_1_name = os.path.join(system_1_directory, "solvated_ligand_1_bound")
system_2_name = os.path.join(system_2_directory, "solvated_ligand_2_bound")

bss.IO.saveMolecules(
    filebase=system_1_name,
    system=system_1_solvated,
    fileformat=["prm7", "rst7"]
)


bss.IO.saveMolecules(
    filebase=system_2_name,
    system=system_2_solvated,
    fileformat=["prm7", "rst7"]
)

['/Users/af25016/projects/ligand_rbfe/data/inputs/protein/bound_ligand_2/solvated_ligand_2_bound.prm7',
 '/Users/af25016/projects/ligand_rbfe/data/inputs/protein/bound_ligand_2/solvated_ligand_2_bound.rst7']